In [ ]:
# -*- coding: utf-8 -*-
# NOTEBOOK POUR TESTER DIFFÉRENTES MÉTHODES DE GESTION DU DÉSÉQUILIBRE

# =============================================================================
# 0. IMPORTS ET CONFIGURATION
# =============================================================================
print("--- Imports et Configuration ---")

import pandas as pd
import numpy as np
import os
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Imports Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    precision_recall_curve,
    auc
)

# Imports Imbalanced-learn (pip install imbalanced-learn)
from imblearn.over_sampling import ADASYN
from imblearn.combine import SMOTEENN

# Imports LightGBM (pip install lightgbm)
try:
    import lightgbm as lgb
    lgbm_available = True
except ImportError:
    lgbm_available = False

# Ignorer les FutureWarning (optionnel)
# warnings.simplefilter(action='ignore', category=FutureWarning)

# --- Configuration ---
DATA_PATH = "/home/kaoutar/Desktop/data/"
# Fichier contenant les 94 features finales (assainies ET enrichies)
INPUT_FILE = os.path.join(DATA_PATH, 'dataset_final_engineered_complet.csv')

# --- CHOIX DE LA MÉTHODE DE GESTION DU DÉSÉQUILIBRE ---
# Options possibles: 'ADASYN', 'SMOTEENN', 'CLASS_WEIGHT', 'NONE' (pour référence)
IMBALANCE_METHOD = 'ADASYN' # <-- MODIFIEZ ICI POUR TESTER UNE AUTRE MÉTHODE
print(f"Méthode de gestion du déséquilibre choisie : {IMBALANCE_METHOD}")
# ------------------------------------------------------


# =============================================================================
# 1. CHARGEMENT DES DONNÉES FINALES
# =============================================================================
print(f"\n--- Chargement du jeu de données final: {INPUT_FILE} ---")

if os.path.exists(INPUT_FILE):
    try:
        df_final = pd.read_csv(INPUT_FILE, sep=',')
        print(f"Données chargées. Shape: {df_final.shape}")

        if 'appetent' not in df_final.columns:
             raise ValueError("Colonne cible 'appetent' manquante.")

        # Préparer X et y
        # S'assurer que X ne contient que des numériques (normalement déjà le cas)
        X = df_final.drop('appetent', axis=1).select_dtypes(include=[np.number])
        y = df_final['appetent']

        # Vérification rapide
        if X.isna().sum().sum() > 0:
            print("ATTENTION: NaNs détectés dans X après chargement. Remplissage par 0...")
            X = X.fillna(0)
        if y.isna().any():
            raise ValueError("NaNs détectés dans la colonne cible y !")

        print(f"Préparation X/y OK. Shape X: {X.shape}, Shape y: {y.shape}")

    except Exception as e:
        print(f"ERREUR lors du chargement ou préparation initiale: {e}")
        X, y = None, None
else:
    print(f"ERREUR: Fichier {INPUT_FILE} non trouvé.")
    X, y = None, None

# =============================================================================
# ÉTAPE V : PRÉPARATION AVANT MODÉLISATION (adaptée à la méthode choisie)
# =============================================================================

if X is not None and y is not None:
    print("\n--- ÉTAPE V: Préparation Avant Modélisation ---")

    # 1. Séparation Train/Test
    print("Séparation Train/Test (80/20)...")
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                        random_state=42, stratify=y)
    print(f"  Shapes - Train: {X_train.shape}, Test: {X_test.shape}")

    # 2. Gestion du Déséquilibre (selon IMBALANCE_METHOD)
    X_train_processed = X_train.copy()
    y_train_processed = y_train.copy()
    apply_resampling = False # Flag pour savoir si on utilise les données resamplées

    if IMBALANCE_METHOD == 'ADASYN':
        print("\nApplication de ADASYN sur les données d'entraînement...")
        adasyn = ADASYN(random_state=42)
        try:
            X_train_processed, y_train_processed = adasyn.fit_resample(X_train.fillna(0), y_train) # fillna sécurité
            print(f"  Shape Train après ADASYN: {X_train_processed.shape}")
            print(f"  Distribution Cible Train après ADASYN:\n{pd.Series(y_train_processed).value_counts(normalize=True)*100}")
            apply_resampling = True
        except Exception as e:
            print(f"  ERREUR pendant ADASYN: {e}. Utilisation des données originales.")
            X_train_processed, y_train_processed = X_train, y_train

    elif IMBALANCE_METHOD == 'SMOTEENN':
        print("\nApplication de SMOTEENN sur les données d'entraînement...")
        smote_enn = SMOTEENN(random_state=42, n_jobs=-1)
        try:
            X_train_processed, y_train_processed = smote_enn.fit_resample(X_train.fillna(0), y_train) # fillna sécurité
            print(f"  Shape Train après SMOTEENN: {X_train_processed.shape}")
            print(f"  Distribution Cible Train après SMOTEENN:\n{pd.Series(y_train_processed).value_counts(normalize=True)*100}")
            apply_resampling = True
        except Exception as e:
            print(f"  ERREUR pendant SMOTEENN: {e}. Utilisation des données originales.")
            X_train_processed, y_train_processed = X_train, y_train

    elif IMBALANCE_METHOD == 'CLASS_WEIGHT':
        print("\nAucun resampling appliqué. Utilisation de class_weight='balanced' dans les modèles.")
        # X_train_processed, y_train_processed restent X_train, y_train

    elif IMBALANCE_METHOD == 'NONE':
        print("\nAucun traitement du déséquilibre appliqué (pour référence).")
        # X_train_processed, y_train_processed restent X_train, y_train

    else: # Inclut SMOTE si spécifié par erreur ou autre valeur
        print("\nMéthode non reconnue ou SMOTE spécifié par défaut. Application de SMOTE...")
        from imblearn.over_sampling import SMOTE # Importer SMOTE ici
        smote = SMOTE(random_state=42)
        try:
            if X_train.isna().sum().sum() > 0: X_train = X_train.fillna(X_train.median())
            X_train_processed, y_train_processed = smote.fit_resample(X_train, y_train)
            print(f"  Shape Train après SMOTE: {X_train_processed.shape}")
            print(f"  Distribution Cible Train après SMOTE:\n{pd.Series(y_train_processed).value_counts(normalize=True)*100}")
            apply_resampling = True
        except Exception as e:
            print(f"  ERREUR pendant SMOTE: {e}. Utilisation données originales.")
            X_train_processed, y_train_processed = X_train, y_train


    # 3. Mise à l'échelle (Scaling)
    print("\nMise à l'échelle (StandardScaler)...")
    numerical_cols_to_scale = X.select_dtypes(include=np.number).columns
    binary_cols = [col for col in X.select_dtypes(include='int').columns if X[col].nunique() <= 2]
    age_group_cols = [col for col in X.columns if col.startswith('age_group_')]
    cols_to_exclude_from_scaling = binary_cols + age_group_cols
    numerical_cols_to_scale = [c for c in numerical_cols_to_scale if c not in cols_to_exclude_from_scaling and c in X_train_processed.columns]

    if numerical_cols_to_scale:
         scaler = StandardScaler()
         X_train_scaled = X_train_processed.copy()
         X_test_scaled = X_test.copy()
         # Remplir NaNs dans Test avant transform (sécurité)
         if X_test_scaled.isna().sum().sum() > 0:
             print("  ATTENTION: NaNs détectés dans X_test avant scaling. Remplissage par médiane de train...")
             test_fill_values = X_train[numerical_cols_to_scale].median()
             X_test_scaled.loc[:, numerical_cols_to_scale] = X_test_scaled[numerical_cols_to_scale].fillna(test_fill_values)

         print(f"  Scaling des colonnes ({len(numerical_cols_to_scale)}): {numerical_cols_to_scale[:5]}...")
         # Fitter sur Train (resampled ou non selon la méthode)
         X_train_scaled.loc[:, numerical_cols_to_scale] = scaler.fit_transform(X_train_scaled[numerical_cols_to_scale])
         # Transformer sur Test
         X_test_scaled.loc[:, numerical_cols_to_scale] = scaler.transform(X_test_scaled[numerical_cols_to_scale])
         print("  Scaling appliqué.")
    else:
         print("  Aucune colonne numérique non-binaire à scaler trouvée.")
         X_train_scaled = X_train_processed.copy()
         X_test_scaled = X_test.copy()

    # Assigner la cible d'entraînement finale (peut être resamplée ou non)
    y_train_final = y_train_processed


# =============================================================================
# ÉTAPE VI : ENTRAÎNEMENT ET ÉVALUATION (avec méthode choisie)
# =============================================================================
print(f"\n--- ÉTAPE VI: Entraînement et Évaluation (Méthode: {IMBALANCE_METHOD}) ---")

if 'X_train_scaled' in locals() and 'y_train_final' in locals():

    models_to_run = {}
    # Définir les modèles en tenant compte de class_weight
    model_configs = {
        "RandomForest": RandomForestClassifier(
            n_estimators=100, random_state=42, n_jobs=-1, max_depth=15, min_samples_leaf=5,
            class_weight='balanced' if IMBALANCE_METHOD == 'CLASS_WEIGHT' else None
        ),
        "LightGBM": lgb.LGBMClassifier(
            objective='binary', random_state=42, n_jobs=-1,
            class_weight='balanced' if IMBALANCE_METHOD == 'CLASS_WEIGHT' else None
            # scale_pos_weight peut être une autre option si IMBALANCE_METHOD == 'SCALE_POS_WEIGHT'
        ) if lgbm_available else None
    }

    final_results = {}

    for name, model in model_configs.items():
        if model is None: continue # Skip si librairie manquante
        print(f"\n--- Modèle : {name} ---")
        try:
            start_time = datetime.now()
            print(f"  Entraînement {name}...")
            # Entraîner sur données potentiellement resamplées, mais SANS class_weight si resamplé
            # Ou sur données originales AVEC class_weight si méthode = CLASS_WEIGHT
            model.fit(X_train_scaled, y_train_final)
            train_time = datetime.now() - start_time
            print(f"  Entraînement terminé en {train_time}")

            print(f"  Évaluation {name} sur Test Set...")
            y_pred_test_model = model.predict(X_test_scaled)
            y_proba_test_model = model.predict_proba(X_test_scaled)[:, 1]

            print(f"\n  Classification Report (Test - {name}):")
            print(classification_report(y_test, y_pred_test_model, zero_division=0))

            print(f"\n  Matrice de Confusion (Test - {name}):")
            cm_model = confusion_matrix(y_test, y_pred_test_model)
            print(cm_model)

            # Calcul métriques clés
            roc_auc = roc_auc_score(y_test, y_proba_test_model) if len(np.unique(y_test)) > 1 else 0.5
            precision, recall, _ = precision_recall_curve(y_test, y_proba_test_model)
            pr_auc = auc(recall, precision)
            report_dict = classification_report(y_test, y_pred_test_model, output_dict=True, zero_division=0)

            print(f"\n  AUC ROC (Test - {name}): {roc_auc:.4f}")
            print(f"  AUC PR (Test - {name}): {pr_auc:.4f}")

            # Stocker pour comparaison future
            final_results[name] = {
                 'AUC_ROC': roc_auc, 'AUC_PR': pr_auc,
                 'F1_Class1': report_dict['1']['f1-score'],
                 'Recall_Class1': report_dict['1']['recall'],
                 'Precision_Class1': report_dict['1']['precision']
            }

        except Exception as e:
            print(f"  ERREUR pour {name}: {e}")
            final_results[name] = {'AUC_ROC': np.nan, 'AUC_PR': np.nan, 'F1_Class1': np.nan}

    # Afficher comparaison finale des méthodes testées si plusieurs existent
    if len(final_results) > 0:
        print("\n\n--- Résultats Finaux du Modèle Testé ---")
        results_final_df = pd.DataFrame(final_results).T
        print(results_final_df)

else:
    print("Erreur: Données non prêtes pour l'étape VI.")

--- Imports et Configuration ---
Méthode de gestion du déséquilibre choisie : ADASYN

--- Chargement du jeu de données final: /home/kaoutar/Desktop/data/dataset_final_engineered_complet.csv ---
Données chargées. Shape: (1051281, 98)
Préparation X/y OK. Shape X: (1051281, 97), Shape y: (1051281,)

--- ÉTAPE V: Préparation Avant Modélisation ---
Séparation Train/Test (80/20)...
  Shapes - Train: (841024, 97), Test: (210257, 97)

Application de ADASYN sur les données d'entraînement...


TypeError: ADASYN.__init__() got an unexpected keyword argument 'n_jobs'